In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install numpy pennylane 
%pip install scikit-learn
%pip install scikit-image

In [19]:
import pennylane as qml
from pennylane import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from skimage.transform import resize

# ----------------------------------------------
#                [퀀텀 컴퓨팅 부]           
# ----------------------------------------------
# [퀀텀 컴퓨팅 부] 양자 회로 정의
n_qubits = 17
# dev = qml.device("default.qubit", wires=n_qubits) # local simulator
dev = qml.device("lightning.qubit", wires=n_qubits) # 느릴 경우 "lightning.qubit"로 변경 가능 (C++ backend)

@qml.qnode(dev)
def quantum_neural_net(inputs, weights):
    """
    양자 회로 정의
    Parameters:
    - inputs: 입력 데이터 (샘플 수, n_qubits)
    - weights: 양자 회로의 파라미터 (샘플 수, n_qubits) - theta

    Returns:
    - qml.expval(qml.PauliZ(0)): 판독 큐비트의 측정 결과 반환
    """
    # 1. 입력 데이터를 양자 회로에 통과
    for i in range(len(inputs)):
        qml.RX(inputs[i] * np.pi, wires=i) # 팁: 픽셀값 0과 1을 라디안 각도로 변환하기 위해 np.pi를 곱해주는 것이 학습에 유리
    
    # 2. 파라미터에 따른 유니터리 연산 (학습부)
    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)

    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])
    
    # 3. 측정 결과를 고전 컴퓨터로 전송
    # 내부적으로 확률적 관측 수행하여 기댓값 도출
    return qml.expval(qml.PauliZ(0))  # 첫 번째 큐비트의 측정 결과 반환


# ----------------------------------------------
#                [고전 컴퓨팅 부]
# ----------------------------------------------

# [고전 컴퓨팅 부] 손실 함수 정의
def loss_function(weights, X, Y):
    """
    손실 함수 정의 (Mean Squared Error)

    Parameters:
    - weights: 양자 회로의 파라미터
    - X: 입력 데이터 (샘플 수, n_qubits)
    - Y: 실제 레이블 (샘플 수)

    Returns:
    - loss: 평균 제곱 오차
    """
    predictions = np.array([quantum_neural_net(x, weights) for x in X])
    loss = np.mean((predictions - Y) ** 2)
    return loss

# [고전 컴퓨팅 부] 데이터셋 전처리(양자화) 함수
def prepare_quantum_dataset(classes=(3,6), img_size=(4,4), test_size=0.2, random_state=42):
    """
    실제 손글씨 숫자 MNIST 데이터셋을 양자 회로에 넣기 전 전처리하는 함수

    Parameters:
    - classes: 이진 분류할 두 개의 숫자 클래스
    - img_size: 축소 이미지 해상도
    - test_size: 테스트 데이터 분리 비율

    Returns:
    - X_train, X_test: [샘플 수, n_qubits] 형태의 이진 픽셀 배열 (0 또는 1)
    - Y_train, Y_test: [샘플 수] 형태의 레이블 배열 (1 또는 -1)
    """
    # 1. 데이터셋 로드
    digits = load_digits()
    X_raw, Y_raw = digits.data, digits.target
    
    # 2. 지정된 클래스(예시: 3과 6)만 필터링
    mask = np.isin(Y_raw, classes)
    X_filtered, Y_filtered = X_raw[mask], Y_raw[mask]
    
    # 3. 레이블 양자 관측값(Pauli-Z)에 맞게 +1, -1로 변환
    Y_binary = np.where(Y_filtered == classes[0], 1.0, -1.0)

    # 4. 이미지 해상도 축소 (8x8 -> 4x4) 및 픽셀 값 정규화 및 이진화
    n_samples = X_filtered.shape[0]
    X_resized = np.zeros((n_samples, img_size[0] * img_size[1]))

    for i in range(n_samples):
        # 4.1. 8x8 이미지를 4x4로 축소
        img = X_filtered[i].reshape(8, 8)
        img_resized = resize(img, img_size, anti_aliasing=True)

        # 4.2. 픽셀 값 정규화 (0~1) 후 이진화 (0 또는 1)
        img_binary = np.where(img_resized > img_resized.mean(), 1.0, 0.0)

        # 4.3. 1차원 배열로 변환하여 저장
        X_resized[i] = img_binary.flatten()

    # 5. 학습용/검증용 데이터 분리
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_resized, Y_binary, test_size=test_size, random_state=random_state
        , stratify=Y_binary # stratify 옵션을 사용하여 클래스 비율 유지
    )

    # 경사하강법이 미분 가능하도록 requires_grad 설정
    X_train = np.array(X_train, requires_grad=False)
    Y_train = np.array(Y_train, requires_grad=False)
    X_test = np.array(X_test, requires_grad=False)
    Y_test = np.array(Y_test, requires_grad=False)

    print("[데이터셋 정보]")
    print(f"이미지 해상도: {img_size[0]}x{img_size[1]}, 입력 큐비트 수: {n_qubits}")
    print(f"학습용 샘플 수: {X_train.shape[0]}, 검증용 샘플 수: {X_test.shape[0]}")

    return X_train, X_test, Y_train, Y_test


# ---------------------------------------------------------
# [실제 수행] 데이터 초기화, 손실 함수, 업데이트 루프
# ---------------------------------------------------------

# 16개의 데이터 큐비트 + 1개 판독 큐비트 = 총 17 큐비트용 데이터셋
X_train, X_test, Y_train, Y_test = prepare_quantum_dataset(classes=(3, 6), img_size=(4, 4))

# 초기 세타 값 설정 (랜덤)
theta = np.random.randn(n_qubits, requires_grad=True)

opt = qml.GradientDescentOptimizer(stepsize=0.1)  # 경사 하강법 최적화기
epochs = 300
batch_size = 5

print("\n학습 시작...")
for epoch in range(epochs):
    # 미니배치 샘플링
    # batch_index = np.random.randint(0, len(X_train), (batch_size,))
    # X_batch = X_train[batch_index]
    # Y_batch = Y_train[batch_index]
    # theta, current_loss = opt.step_and_cost(lambda w: loss_function(w, X_batch, Y_batch), theta)    

    # 손실 함수 계산 & 파라미터 업데이트 - 페니레인 옵티마이저에 맞는 루프 호출법
    theta, current_loss = opt.step_and_cost(lambda w: loss_function(w, X_train, Y_train), theta)    

    if (epoch+1) % 5 == 0:
        print(f"Epoch: {epoch+1:3d} | Loss: {current_loss:.4f}")

print("학습 완료!")
print("최적화된 theta 값:", theta)

# 테스트 데이터셋으로 테스트
Y_test_pred = np.array([quantum_neural_net(x, theta) for x in X_test])
predicted_labels = np.where(Y_test_pred > 0, 1, -1)
accuracy = np.mean(predicted_labels == Y_test)
print(f"테스트 정확도: {accuracy:.4f}")

: 